# Warranty Review Analysis

Detect warranty-related reviews using a keyword baseline and explore related terms with Word2Vec.

Dataset: `Train_subset_50k` (team-preprocessed Amazon reviews).

In [7]:

from pathlib import Path
import zipfile

cwd = Path.cwd().resolve()
root = next(
    (p for p in [cwd, *cwd.parents] if (p / "src" / "nlp").is_dir()),
    None
)

print("Repository:", root)

if root is not None:
    data_dirs = [root / "data", root.parent / "data"]

    for folder in data_dirs:
        print(f"\n{folder}")
        if folder.exists():
            for file in folder.iterdir():
                print(" -", file.name)
        else:
            print("Directory not found")


Repository: C:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer

C:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\data

C:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\data


In [8]:
from pathlib import Path
import zipfile

search_dirs = [
    root / "data",
    root.parent / "data",
    Path.home() / "Downloads"
]

datasets = [
    file
    for folder in search_dirs if folder.exists()
    for file in folder.glob("*.zip")
    if "train" in file.name.lower() or "subset" in file.name.lower()
]

for file in datasets:
    print("Found:", file)
    with zipfile.ZipFile(file) as z:
        print("Contents:", z.namelist()[:10])

if not datasets:
    print("Dataset not found. Download train subset.zip first.")

Found: C:\Users\HP\Downloads\train subset.zip
Contents: ['train subset/', 'train subset/Train_subset_50k.csv']


In [9]:
import csv
import io

zip_path = datasets[0]

with zipfile.ZipFile(zip_path) as z:
    csv_file = next(name for name in z.namelist() if name.endswith(".csv"))

    with z.open(csv_file) as f:
        reader = csv.DictReader(io.TextIOWrapper(f, encoding="utf-8-sig"))
        reviews = list(reader)

print("Total reviews:", len(reviews))
print("Columns:", list(reviews[0].keys()))
print("Sample review:", reviews[0]["reviewText_clean"][:250])

Total reviews: 50000
Columns: ['overall', 'vote', 'verified', 'reviewTime', 'reviewerID', 'asin', 'style', 'reviewerName', 'reviewText', 'summary', 'unixReviewTime', 'review_year', 'reviewText_clean', 'full_text']
Sample review: Quality wire. Looks nice and works just as expected! I used this to connect my phone/tablet to vintage stereo receiver for streaming Prime Music. Worked great. I have since hooked it up to my newly acquired Amazon Dot for same purpose.


## 1. Warranty Keyword Baseline

Find reviews mentioning "warranty" or "guarantee" using regex.
Compare matches in cleaned reviews and full text.

In [10]:
import re

pattern = re.compile(r"\b(?:warranty|guarantee)\b", re.IGNORECASE)

for review in reviews:
    review["clean_match"] = bool(
        pattern.search(review.get("reviewText_clean") or "")
    )
    review["full_match"] = bool(
        pattern.search(review.get("full_text") or "")
    )

clean_matches = [r for r in reviews if r["clean_match"]]
full_matches = [r for r in reviews if r["full_match"]]

print("Total reviews:", len(reviews))
print("Clean text matches:", len(clean_matches))
print("Full text matches:", len(full_matches))
print("Additional matches:", sum(
    r["full_match"] and not r["clean_match"] for r in reviews
))

print("\nSample matches:")
for review in full_matches[:3]:
    print("-", (review.get("full_text") or "")[:200])

Total reviews: 50000
Clean text matches: 965
Full text matches: 979
Additional matches: 14

Sample matches:
- Make sure you have a low profile (LP) slot If you need a low profile, serial adapter for a desktop computer, this 9 Pin Serial Male to 10 Pin Motherboard Header card from StarTech is a solid and inexp
- Great Sound & Lots Of Features and Uses! &nbsp;I saw something very similar to this in a store chain last year and wanted to purchase it and never did so when I saw this one, I decided it was time. I 
- Was Great While It Worked! Worked great for about 2.5 months, then it started malfunctioning. It constantly would give a low battery signal, even after receiving multiple 12 hour charges and the light


### Checking Additional Matches

Inspect reviews found only after including the summary.

In [11]:
extra_matches = [
    r for r in reviews
    if r["full_match"] and not r["clean_match"]
]

print("Additional reviews:", len(extra_matches))

for i, review in enumerate(extra_matches[:5], 1):
    text = review.get("full_text") or ""
    match = pattern.search(text)

    start = max(0, match.start() - 70)
    end = min(len(text), match.end() + 100)

    print(f"\n{i}. Rating: {review['overall']}")
    print("Summary:", review["summary"])
    print("Context:", text[start:end])

Additional reviews: 14

1. Rating: 5
Summary: Great for speed, worry free warranty!
Context: Great for speed, worry free warranty! Top of the line, included software works great! Installed on my PCB designer computer at work and 

2. Rating: 1
Summary: It works fine but it will fail on you before the 2 year warranty
Context: It works fine but it will fail on you before the 2 year warranty Died after 4 months of purchashed from fry''s called cs to see if I could get it replaced. After fi

3. Rating: 2
Summary: Bought for the guarantee against sweat failure. Failed in  less than 30 days.
Context: Bought for the guarantee against sweat failure. Failed in  less than 30 days. I'm updating this review because these headpho

4. Rating: 3
Summary: How to get this repaired or ck for warranty
Context: How to get this repaired or ck for warranty I bought this for my son, for his XBox One, in December 2016. Last year, after 6 months, it stopped

5. Rating: 5
Summary: Works and looks like new, 1 ye

### Baseline Results

979 out of 50,000 reviews matched the warranty keywords.
Using full_text captures 14 additional reviews.

In [12]:
import csv
import json

output_dir = root / "outputs_warranty"
output_dir.mkdir(exist_ok=True)

columns = ["asin", "overall", "summary", "reviewText_clean", "full_text"]

with open(output_dir / "day4_warranty_matches.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    writer.writerows({key: r.get(key, "") for key in columns} for r in full_matches)

summary = {
    "total_reviews": len(reviews),
    "clean_matches": len(clean_matches),
    "full_text_matches": len(full_matches),
    "additional_matches": len(extra_matches),
    "keywords": ["warranty", "guarantee"]
}

(output_dir / "day4_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("Saved:", output_dir / "day4_warranty_matches.csv")
print("Saved:", output_dir / "day4_summary.json")

Saved: C:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\outputs_warranty\day4_warranty_matches.csv
Saved: C:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\outputs_warranty\day4_summary.json


In [13]:
import importlib.util

print("gensim:", "available" if importlib.util.find_spec("gensim") else "not installed")
print("numpy:", "available" if importlib.util.find_spec("numpy") else "not installed")
print("scipy:", "available" if importlib.util.find_spec("scipy") else "not installed")

gensim: available
numpy: available
scipy: available


In [14]:

from collections import defaultdict
import csv

# Warranty review stats by product
products = defaultdict(lambda: {"count": 0, "rating_sum": 0})

for review in full_matches:
    asin = review.get("asin", "").strip()
    try:
        rating = float(review["overall"])
    except (ValueError, TypeError):
        continue

    if asin:
        products[asin]["count"] += 1
        products[asin]["rating_sum"] += rating

product_rows = [
    {
        "asin": asin,
        "warranty_reviews": stats["count"],
        "avg_rating": round(stats["rating_sum"] / stats["count"], 2)
    }
    for asin, stats in products.items()
]

product_rows.sort(key=lambda x: x["warranty_reviews"], reverse=True)

with open(output_dir / "day4_product_stats.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["asin", "warranty_reviews", "avg_rating"])
    writer.writeheader()
    writer.writerows(product_rows)

print("Products analyzed:", len(product_rows))
print("Top products:")
for row in product_rows[:5]:
    print(row)

# Check previous Word2Vec results
print("\nWord2Vec files:")
files = sorted(output_dir.glob("*word2vec*"))

for file in files:
    print("-", file.name)

candidate_file = output_dir / "day5_word2vec_candidates_REVIEW_REQUIRED.csv"

if candidate_file.exists():
    with candidate_file.open(encoding="utf-8-sig", newline="") as f:
        candidates = list(csv.DictReader(f))

    print("\nCandidates:", len(candidates))
    print("Columns:", list(candidates[0]) if candidates else [])
    for row in candidates[:20]:
        print(row)
else:
    print("\nNo saved Word2Vec candidates found.")


Products analyzed: 903
Top products:
{'asin': 'B00LF10KTE', 'warranty_reviews': 3, 'avg_rating': 5.0}
{'asin': 'B01DEMTOQ6', 'warranty_reviews': 3, 'avg_rating': 3.67}
{'asin': 'B01A6PGZ70', 'warranty_reviews': 3, 'avg_rating': 5.0}
{'asin': 'B01E3XLNA0', 'warranty_reviews': 3, 'avg_rating': 4.67}
{'asin': 'B013HNYVCE', 'warranty_reviews': 3, 'avg_rating': 3.33}

Word2Vec files:
- day5_word2vec_candidates_REVIEW_REQUIRED.csv

Candidates: 70
Columns: ['seed', 'candidate', 'cosine_similarity', 'human_decision', 'review_reason']
{'seed': 'warranty', 'candidate': 'service', 'cosine_similarity': '0.6198', 'human_decision': 'PENDING', 'review_reason': ''}
{'seed': 'warranty', 'candidate': 'shipping', 'cosine_similarity': '0.5993', 'human_decision': 'PENDING', 'review_reason': ''}
{'seed': 'warranty', 'candidate': 'policy', 'cosine_similarity': '0.5964', 'human_decision': 'PENDING', 'review_reason': ''}
{'seed': 'warranty', 'candidate': 'later', 'cosine_similarity': '0.5935', 'human_decision'

## 2. Word2Vec Candidate Review

Check suggested terms against real reviews before expanding the baseline.

In [15]:
from collections import Counter
import re
import csv

baseline_ids = {
    i for i, review in enumerate(reviews)
    if review["full_match"]
}

results = []

for candidate in candidates:
    word = candidate["candidate"].strip().lower()
    word_pattern = re.compile(r"\b" + re.escape(word) + r"\b", re.I)

    matches = []
    new_matches = []

    for i, review in enumerate(reviews):
        text = review.get("full_text") or ""
        match = word_pattern.search(text)

        if match:
            matches.append(i)

            if i not in baseline_ids:
                start = max(0, match.start() - 70)
                end = min(len(text), match.end() + 100)

                new_matches.append(text[start:end].replace("\n", " "))

    results.append({
        "seed": candidate["seed"],
        "candidate": word,
        "cosine_similarity": candidate["cosine_similarity"],
        "total_matches": len(matches),
        "new_matches": len(new_matches),
        "example_context": new_matches[0] if new_matches else "",
        "human_decision": "PENDING"
    })

audit_path = output_dir / "day5_candidate_audit.csv"

with audit_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print("Word2Vec suggestions:", len(results))
print("Unique candidates:", len({r["candidate"] for r in results}))
print("Saved:", audit_path)

print("\nCandidates with new review matches:\n")

seen = set()

for row in sorted(
    results,
    key=lambda r: int(r["new_matches"]),
    reverse=True
):
    if row["candidate"] in seen:
        continue

    seen.add(row["candidate"])

    print(
        f"{row['candidate']:16} "
        f"new={row['new_matches']:5} "
        f"similarity={row['cosine_similarity']}"
    )

    if len(seen) == 20:
        break

Word2Vec suggestions: 70
Unique candidates: 68
Saved: C:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\outputs_warranty\day5_candidate_audit.csv

Candidates with new review matches:

last             new= 2798 similarity=0.4988
support          new= 1890 similarity=0.4859
replacement      new= 1754 similarity=0.5332
service          new= 1556 similarity=0.6198
customer         new= 1330 similarity=0.5767
company          new= 1240 similarity=0.5261
ago              new= 1234 similarity=0.5364
later            new=  989 similarity=0.5935
past             new=  943 similarity=0.5052
seller           new=  824 similarity=0.5425
shipping         new=  601 similarity=0.5993
lasted           new=  579 similarity=0.5789
contacted        new=  553 similarity=0.5048
manufacturer     new=  534 similarity=0.5085
defective        new=  507 similarity=0.499
refund           new=  418 similarity=0.5007
shipped          new=  342 similarity=0.4989
anker            new=  3

### Candidate Validation

Inspect real review contexts for potentially useful warranty terms.

In [16]:
check_words = [
    "policy", "lifetime", "expired", "rma",
    "replacement", "service", "support", "refund"
]

for word in check_words:
    regex = re.compile(r"\b" + re.escape(word) + r"\b", re.I)

    examples = []
    for review in reviews:
        if review["full_match"]:
            continue

        text = review.get("full_text") or ""
        match = regex.search(text)

        if match:
            start = max(0, match.start() - 80)
            end = min(len(text), match.end() + 100)
            examples.append(text[start:end].replace("\n", " "))

        if len(examples) == 2:
            break

    print(f"\n{word.upper()}")
    for example in examples:
        print("-", example)


POLICY
- ich happens). It was bought from another website that was horrible in returning policy. Luckily enough Buffalo stepped in and cleared everything. This is the ONLY backup drive that I kno
- Bought with a remanufactured computer as a "insurance policy" against ... Bought with a remanufactured computer as a "insurance policy" against unexpected failu

LIFETIME
- ver go on a trip without it! We upgraded from an older Garmin because we wanted lifetime maps. Love the 5" display and really have enjoyed the traffic information also. Other features we r
-  Quality of materials I would give a 7 out of 10. I don't expect this to last a lifetime but for the pricepoint I can't complain. I like that it has a lot of storage pockets but I would ha

EXPIRED
- oth of my HP 14 Chromebooks died in just over a year (just after the warranties expired). One one the fan is shot; on the other (which is fanless), the keyboard died. I'm fed up with the 
- i's I have purchased in the past that died just

### Keyword Screening

Review Word2Vec suggestions and flag ambiguous terms before using them for matching.

In [17]:
import csv
from collections import Counter

context_dependent = {
    "rma", "expired", "policy", "lifetime",
    "replacement", "service", "support",
    "refund", "coverage", "manufacturer",
    "honor", "seller", "vendor", "contacted",
    "defective", "customer", "rep"
}

reject = {
    "last", "ago", "later", "past", "shipping",
    "shipped", "company", "date", "period",
    "lasted", "anker"
}

screened = []

for row in results:
    word = row["candidate"]

    if word in reject:
        decision = "reject"
        reason = "Too broad for standalone matching"
    elif word in context_dependent:
        decision = "context-dependent"
        reason = "Check warranty context before matching"
    else:
        decision = "review"
        reason = "Needs manual inspection"

    screened.append({
        **row,
        "screening_decision": decision,
        "review_reason": reason
    })

path = output_dir / "day5_screened_candidates.csv"

with path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=screened[0].keys())
    writer.writeheader()
    writer.writerows(screened)

print("Total candidates:", len(screened))
print("Unique words:", len(set(r["candidate"] for r in screened)))
print("Screening:", dict(Counter(r["screening_decision"] for r in screened)))
print("Saved:", path)

Total candidates: 70
Unique words: 68
Screening: {'context-dependent': 17, 'reject': 11, 'review': 42}
Saved: C:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\outputs_warranty\day5_screened_candidates.csv


### Baseline Check

Check where the additional keyword matches came from.

In [18]:
summary_only = sum(
    bool(pattern.search(r.get("summary") or ""))
    for r in extra_matches
)

print("Total reviews:", len(reviews))
print("Baseline matches:", len(full_matches))
print("Additional matches:", len(extra_matches))
print("Additional matches with keyword in summary:", summary_only)
print("Products:", len(product_rows))

Total reviews: 50000
Baseline matches: 979
Additional matches: 14
Additional matches with keyword in summary: 14
Products: 903


### Word2Vec Shortlist

Keep one entry per candidate and prioritize context-dependent terms for manual review.

In [19]:
import csv

priority = {
    "rma", "expired", "policy", "lifetime",
    "replacement", "service", "support",
    "refund", "coverage", "manufacturer",
    "honor", "seller", "vendor", "contacted",
    "defective", "customer", "rep"
}

unique = {}

for row in screened:
    word = row["candidate"]

    if word not in unique or float(row["cosine_similarity"]) > float(unique[word]["cosine_similarity"]):
        unique[word] = row

shortlist = sorted(
    (unique[w] for w in priority if w in unique),
    key=lambda x: float(x["cosine_similarity"]),
    reverse=True
)

shortlist_path = output_dir / "day5_review_shortlist.csv"

with shortlist_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=shortlist[0].keys())
    writer.writeheader()
    writer.writerows(shortlist)

print("Unique Word2Vec candidates:", len(unique))
print("Priority terms:", len(shortlist))
print("Saved:", shortlist_path)

for row in shortlist:
    print(row["candidate"], row["cosine_similarity"])

Unique Word2Vec candidates: 68
Priority terms: 16
Saved: C:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\outputs_warranty\day5_review_shortlist.csv
honor 0.7467
service 0.6198
policy 0.5964
customer 0.5767
rep 0.566
lifetime 0.5614
expired 0.5574
rma 0.5519
seller 0.5425
vendor 0.5412
replacement 0.5332
manufacturer 0.5085
contacted 0.5048
refund 0.5007
defective 0.499
support 0.4859


## Results

The keyword baseline identifies direct warranty mentions.
Word2Vec provides additional candidate terms for manual review.

In [20]:
import json

report = {
    "dataset": "Train_subset_50k.csv",
    "total_reviews": len(reviews),
    "baseline_keywords": ["warranty", "guarantee"],
    "clean_text_matches": len(clean_matches),
    "full_text_matches": len(full_matches),
    "additional_summary_matches": len(extra_matches),
    "products_analyzed": len(product_rows),
    "word2vec_candidates": len(candidates),
    "unique_candidates": len(unique),
    "priority_terms": len(shortlist),
    "validation_status": "Preliminary screening only",
    "precision_recall": "Not measured"
}

report_path = output_dir / "warranty_final_summary.json"
report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

docs_dir = root / "docs"
docs_dir.mkdir(exist_ok=True)

notes = f"""# Warranty Review Analysis

## Dataset
Train_subset_50k.csv — {len(reviews):,} preprocessed Amazon reviews.

## Day 4 — Keyword Baseline
- Keywords: warranty, guarantee
- Clean text matches: {len(clean_matches)}
- Full text matches: {len(full_matches)}
- Additional matches: {len(extra_matches)}
- Products with matched reviews and valid ratings: {len(product_rows)}

Full text is used for matching to include review summaries.

## Day 5 — Word2Vec
- Precomputed candidate entries: {len(candidates)}
- Unique candidate terms: {len(unique)}
- Priority terms for further review: {len(shortlist)}

Candidate similarities come from an earlier Word2Vec experiment.
The model was not retrained in this notebook.

## Limitations
- Keyword matches are not manually verified labels.
- Word similarity does not guarantee warranty relevance.
- Candidate screening is preliminary.
- Precision and recall have not been measured.
- Average product ratings describe overall reviews, not warranty sentiment.

## Outputs
- day4_warranty_matches.csv
- day4_product_stats.csv
- day4_summary.json
- day5_candidate_audit.csv
- day5_screened_candidates.csv
- day5_review_shortlist.csv
- warranty_final_summary.json
"""

notes_path = docs_dir / "WARRANTY_BASELINE_NOTES.md"
notes_path.write_text(notes, encoding="utf-8")

print("Report saved:", report_path)
print("Notes saved:", notes_path)
print("\nSummary:")
print(json.dumps(report, indent=2))

Report saved: C:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\outputs_warranty\warranty_final_summary.json
Notes saved: C:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\docs\WARRANTY_BASELINE_NOTES.md

Summary:
{
  "dataset": "Train_subset_50k.csv",
  "total_reviews": 50000,
  "baseline_keywords": [
    "warranty",
    "guarantee"
  ],
  "clean_text_matches": 965,
  "full_text_matches": 979,
  "additional_summary_matches": 14,
  "products_analyzed": 903,
  "word2vec_candidates": 70,
  "unique_candidates": 68,
  "priority_terms": 16,
  "validation_status": "Preliminary screening only",
  "precision_recall": "Not measured"
}


### Co-occurrence Analysis

Compare Word2Vec candidates with their observed association with warranty mentions.

In [21]:
from collections import Counter
import re
import math

words = {r["candidate"].lower() for r in candidates}
word_counts = Counter()
joint_counts = Counter()

for review in reviews:
    text = (review.get("full_text") or "").lower()
    tokens = set(re.findall(r"\b[a-z]+\b", text))
    present = tokens & words

    for word in present:
        word_counts[word] += 1
        if review["full_match"]:
            joint_counts[word] += 1

baseline_rate = len(full_matches) / len(reviews)

comparison = []

for word in words:
    total = word_counts[word]
    joint = joint_counts[word]

    if total < 10:
        continue

    lift = (joint / total) / baseline_rate

    comparison.append({
        "word": word,
        "total": total,
        "with_warranty": joint,
        "lift": round(lift, 2)
    })

comparison.sort(key=lambda x: x["lift"], reverse=True)

for row in comparison[:20]:
    print(row)

{'word': 'guarantee', 'total': 92, 'with_warranty': 92, 'lift': 51.07}
{'word': 'honor', 'total': 36, 'with_warranty': 23, 'lift': 32.63}
{'word': 'expired', 'total': 57, 'with_warranty': 30, 'lift': 26.88}
{'word': 'lifetime', 'total': 164, 'with_warranty': 85, 'lift': 26.47}
{'word': 'discuss', 'total': 13, 'with_warranty': 3, 'lift': 11.79}
{'word': 'rma', 'total': 121, 'with_warranty': 26, 'lift': 10.97}
{'word': 'rep', 'total': 87, 'with_warranty': 13, 'lift': 7.63}
{'word': 'contacted', 'total': 648, 'with_warranty': 95, 'lift': 7.49}
{'word': 'policy', 'total': 114, 'with_warranty': 16, 'lift': 7.17}
{'word': 'warrantee', 'total': 15, 'with_warranty': 2, 'lift': 6.81}
{'word': 'period', 'total': 361, 'with_warranty': 46, 'lift': 6.51}
{'word': 'preface', 'total': 16, 'with_warranty': 2, 'lift': 6.38}
{'word': 'emailed', 'total': 114, 'with_warranty': 14, 'lift': 6.27}
{'word': 'satisfaction', 'total': 84, 'with_warranty': 10, 'lift': 6.08}
{'word': 'transaction', 'total': 18, 'w

In [22]:
path = output_dir / "day5_cooccurrence_comparison.csv"

with path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=comparison[0].keys())
    writer.writeheader()
    writer.writerows(comparison)

print("Saved:", path)
print("Words analyzed:", len(comparison))

Saved: C:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\outputs_warranty\day5_cooccurrence_comparison.csv
Words analyzed: 61


### Spelling Variants

Check common warranty word variants missed by the baseline.

In [23]:
variant_pattern = re.compile(
    r"\b(?:warranties|warrantee|warrantees|guarantees)\b",
    re.I
)

variant_rows = []

for review in reviews:
    if review["full_match"]:
        continue

    text = review.get("full_text") or ""
    match = variant_pattern.search(text)

    if match:
        variant_rows.append({
            "asin": review["asin"],
            "matched_term": match.group(),
            "context": text[max(0, match.start()-80):match.end()+100]
        })

path = output_dir / "day5_variant_candidates.csv"

with path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["asin", "matched_term", "context"])
    writer.writeheader()
    writer.writerows(variant_rows)

print("Additional variant matches:", len(variant_rows))
print("Breakdown:", dict(Counter(r["matched_term"].lower() for r in variant_rows)))
print("Saved:", path)

for row in variant_rows[:5]:
    print("-", row["context"][:160])

Additional variant matches: 26
Breakdown: {'guarantees': 3, 'warranties': 10, 'warrantee': 13}
Saved: C:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\outputs_warranty\day5_variant_candidates.csv
- ecommend the caddy because it does an excellent job of organizing batteries and guarantees they won't pop out.
- ollars on will last long enough to make the purchase worth it. The Square Trade warranties offered through Amazon are great because I no longer have to worry ab
- ew years about there being more resistance on Squaretrade's part to honor their warranties and I hope that is not the case. I continue to purchase them, occasio
- the time. Both of my HP 14 Chromebooks died in just over a year (just after the warranties expired). One one the fan is shot; on the other (which is fanless), t
- Another reason I like this over the Yamahas is that this one is UL Listed. That guarantees a certain minimum level of internal component quality. (transformers,


## Final Results

The regex baseline is complete. Word2Vec candidates were screened using review contexts and co-occurrence statistics. Manual validation is still required.

In [24]:
import json

top_terms = [
    row for row in comparison
    if row["word"] not in {"warranty", "guarantee"}
][:10]

report = {
    "dataset": "Train_subset_50k.csv",
    "reviews": len(reviews),
    "baseline_clean": len(clean_matches),
    "baseline_full_text": len(full_matches),
    "additional_summary_matches": len(extra_matches),
    "products": len(product_rows),
    "word2vec_suggestions": len(candidates),
    "unique_candidates": len({r["candidate"] for r in candidates}),
    "screening": dict(Counter(r["screening_decision"] for r in screened)),
    "cooccurrence_terms_analyzed": len(comparison),
    "additional_variant_candidates": len(variant_rows),
    "precision_recall": "Not evaluated"
}

(output_dir / "warranty_final_summary.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

docs_dir = root / "docs"
docs_dir.mkdir(exist_ok=True)

notes = f"""# Warranty Review Analysis

## Data
50,000 preprocessed Amazon reviews from Train_subset_50k.csv.

## Keyword baseline
- Keywords: warranty, guarantee
- Clean text matches: {len(clean_matches)}
- Full text matches: {len(full_matches)}
- Additional summary matches: {len(extra_matches)}
- Products with matched reviews: {len(product_rows)}

## Word2Vec
- 70 saved suggestions, 68 unique terms
- Candidates screened using review examples
- 61 terms included in the co-occurrence comparison
- Co-occurrence lift is not a relevance score
- Seed keywords must be excluded from lift-based ranking

## Spelling variants
- {len(variant_rows)} additional candidate reviews
- Requires manual inspection

## Limitations
- Word2Vec candidates were loaded from an earlier experiment.
- Word2Vec was not retrained in this notebook.
- Screening is preliminary, not fully validated.
- Precision and recall have not been measured.
- Product ratings are overall ratings, not warranty-specific sentiment.
- FastText, GloVe and BERT were not evaluated.

## Next steps
- Manually label a review sample
- Validate candidate keywords and spelling variants
- Compare precision and recall against the baseline
"""

notes_path = docs_dir / "WARRANTY_BASELINE_NOTES.md"
notes_path.write_text(notes, encoding="utf-8")

print(json.dumps(report, indent=2))
print("Saved:", notes_path)

{
  "dataset": "Train_subset_50k.csv",
  "reviews": 50000,
  "baseline_clean": 965,
  "baseline_full_text": 979,
  "additional_summary_matches": 14,
  "products": 903,
  "word2vec_suggestions": 70,
  "unique_candidates": 68,
  "screening": {
    "context-dependent": 17,
    "reject": 11,
    "review": 42
  },
  "cooccurrence_terms_analyzed": 61,
  "additional_variant_candidates": 26,
  "precision_recall": "Not evaluated"
}
Saved: C:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\docs\WARRANTY_BASELINE_NOTES.md


## Model Comparison

Build a review sample to evaluate keyword and embedding-based methods.

In [25]:
import random
import csv
import re

random.seed(42)

candidate_terms = [
    "rma", "expired", "warrantee", "warranties",
    "replacement", "policy", "coverage", "refund"
]

candidate_regex = re.compile(
    r"\b(?:" + "|".join(candidate_terms) + r")\b",
    re.I
)

baseline_pool = [
    i for i, r in enumerate(reviews)
    if r["full_match"]
]

candidate_pool = [
    i for i, r in enumerate(reviews)
    if not r["full_match"]
    and candidate_regex.search(r.get("full_text") or "")
]

other_pool = [
    i for i, r in enumerate(reviews)
    if not r["full_match"]
    and not candidate_regex.search(r.get("full_text") or "")
]

selected = (
    [(i, "baseline") for i in random.sample(baseline_pool, 30)]
    + [(i, "candidate") for i in random.sample(candidate_pool, 30)]
    + [(i, "other") for i in random.sample(other_pool, 30)]
)

random.shuffle(selected)

evaluation = [
    {
        "review_id": i,
        "group": group,
        "text": reviews[i].get("full_text") or "",
        "label": ""
    }
    for i, group in selected
]

eval_path = output_dir / "model_evaluation_sample.csv"

with eval_path.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=evaluation[0].keys())
    writer.writeheader()
    writer.writerows(evaluation)

print("Evaluation reviews:", len(evaluation))
print("Baseline pool:", len(baseline_pool))
print("Candidate pool:", len(candidate_pool))
print("Saved:", eval_path)

Evaluation reviews: 90
Baseline pool: 979
Candidate pool: 2487
Saved: C:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\outputs_warranty\model_evaluation_sample.csv


In [26]:
for row in evaluation[:5]:
    print("ID:", row["review_id"])
    print("Group:", row["group"])
    print("Text:", row["text"][:350])
    print("-" * 50)

ID: 44486
Group: candidate
Text: Holy Microphonics Batman! This cable gets the job done, but it has especially bad microphonics (ie, annoying noise when the cable rubs against something). The microphonics are unusually bad with this cable for two reasons: - the cable is more rigid / stiff than most - the cable has a very coarse/rough shielding So while I think this will be a durab
--------------------------------------------------
ID: 43587
Group: other
Text: Good vslue and decent product Great buy for the money exactly as ordered. One drawback is that despite being compatible with my phone "galaxy s5" it occasionally throws up an error. It's a quick fix, just open the setting unmount then remount the sd card
--------------------------------------------------
ID: 10829
Group: baseline
Text: ... earlier then expected so we are off to a great start. The cable itself is very well made ... Received item earlier then expected so we are off to a great start. The cable itself is very well mad

In [27]:
%pip -q install gensim sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import gensim
import numpy as np
from sentence_transformers import SentenceTransformer

print("gensim:", gensim.__version__)
print("numpy:", np.__version__)
print("SentenceTransformer: ready")

c:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


gensim: 4.4.0
numpy: 2.5.4
SentenceTransformer: ready


In [29]:
import csv

eval_path = output_dir / "model_evaluation_sample.csv"

with eval_path.open(encoding="utf-8-sig", newline="") as f:
    eval_rows = list(csv.DictReader(f))

print("Training reviews:", len(reviews))
print("Evaluation reviews:", len(eval_rows))

Training reviews: 50000
Evaluation reviews: 90


In [30]:
import re
from gensim.models import FastText

sentences = [
    re.findall(r"[a-z]+", (r.get("full_text") or "").lower())
    for r in reviews[:20000]
]

fasttext = FastText(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=3,
    sg=0,
    workers=2,
    epochs=5,
    seed=42
)

ft_rows = []

for seed in ["warranty", "guarantee"]:
    for word, score in fasttext.wv.most_similar(seed, topn=30):
        ft_rows.append({
            "seed": seed,
            "candidate": word,
            "similarity": round(float(score), 4)
        })

ft_path = output_dir / "day5_fasttext_candidates.csv"

with ft_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=ft_rows[0].keys())
    writer.writeheader()
    writer.writerows(ft_rows)

print("FastText candidates:", len(ft_rows))
print("Unique words:", len({r["candidate"] for r in ft_rows}))
print("Saved:", ft_path)
print("\nTop candidates:")
for row in ft_rows[:10]:
    print(row)

FastText candidates: 60
Unique words: 50
Saved: C:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\outputs_warranty\day5_fasttext_candidates.csv

Top candidates:
{'seed': 'warranty', 'candidate': 'warrantee', 'similarity': 0.9646}
{'seed': 'warranty', 'candidate': 'warrant', 'similarity': 0.9512}
{'seed': 'warranty', 'candidate': 'warrants', 'similarity': 0.9103}
{'seed': 'warranty', 'candidate': 'warranties', 'similarity': 0.8894}
{'seed': 'warranty', 'candidate': 'warrenty', 'similarity': 0.8473}
{'seed': 'warranty', 'candidate': 'warrior', 'similarity': 0.7837}
{'seed': 'warranty', 'candidate': 'guarantee', 'similarity': 0.7368}
{'seed': 'warranty', 'candidate': 'wary', 'similarity': 0.7287}
{'seed': 'warranty', 'candidate': 'seller', 'similarity': 0.7115}
{'seed': 'warranty', 'candidate': 'refunded', 'similarity': 0.6959}


In [31]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

queries = [
    "The customer discusses a product warranty or guarantee.",
    "The product warranty expired or does not cover the damage.",
    "The customer requests repair or replacement under warranty."
]

texts = [row["text"] for row in eval_rows]

text_vectors = model.encode(texts, normalize_embeddings=True)
query_vectors = model.encode(queries, normalize_embeddings=True)

scores = (text_vectors @ query_vectors.T).max(axis=1)

for row, score in zip(eval_rows, scores):
    row["sbert_score"] = round(float(score), 4)
    row["regex_prediction"] = int(bool(pattern.search(row["text"])))

scores_path = output_dir / "day5_sbert_scores.csv"

with scores_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=eval_rows[0].keys())
    writer.writeheader()
    writer.writerows(eval_rows)

print("Reviews scored:", len(eval_rows))
print("Saved:", scores_path)

for row in eval_rows[:5]:
    print(row["review_id"], row["regex_prediction"], row["sbert_score"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2130.26it/s]


Reviews scored: 90
Saved: C:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\outputs_warranty\day5_sbert_scores.csv
44486 0 0.0304
43587 0 0.1698
10829 1 0.2912
18811 0 0.0168
37606 1 0.2879


### Model Predictions

Compare keyword-based predictions from Word2Vec and FastText with the regex baseline.

In [32]:
import re
import csv
from collections import defaultdict

def top_terms(rows, score_key, n=15):
    groups = defaultdict(list)

    for row in rows:
        groups[row["seed"]].append(row)

    terms = {"warranty", "guarantee"}

    for seed, group in groups.items():
        ranked = sorted(
            group,
            key=lambda r: float(r[score_key]),
            reverse=True
        )
        terms.update(r["candidate"].lower() for r in ranked[:n])

    return terms

w2v_terms = top_terms(candidates, "cosine_similarity")
ft_terms = top_terms(ft_rows, "similarity")

def make_pattern(words):
    return re.compile(
        r"\b(?:" + "|".join(re.escape(w) for w in sorted(words)) + r")\b",
        re.I
    )

w2v_pattern = make_pattern(w2v_terms)
ft_pattern = make_pattern(ft_terms)

for row in eval_rows:
    text = row["text"]
    row["w2v_prediction"] = int(bool(w2v_pattern.search(text)))
    row["fasttext_prediction"] = int(bool(ft_pattern.search(text)))
    row["sbert_prediction"] = int(float(row["sbert_score"]) >= 0.30)

print("Regex:", sum(int(r["regex_prediction"]) for r in eval_rows))
print("Word2Vec:", sum(r["w2v_prediction"] for r in eval_rows))
print("FastText:", sum(r["fasttext_prediction"] for r in eval_rows))
print("Sentence-BERT:", sum(r["sbert_prediction"] for r in eval_rows))

Regex: 30
Word2Vec: 56
FastText: 43
Sentence-BERT: 12


### Manual Evaluation

Label a small review sample to compare prediction quality.

In [33]:
from pathlib import Path
import csv
import random

cwd = Path.cwd().resolve()
root = next(
    p for p in [cwd, *cwd.parents]
    if (p / "outputs_warranty").is_dir()
)

output_dir = root / "outputs_warranty"

with (output_dir / "day5_sbert_scores.csv").open(
    encoding="utf-8-sig", newline=""
) as f:
    eval_rows = list(csv.DictReader(f))

rng = random.Random(42)
sample = []

for group in ["baseline", "candidate", "other"]:
    pool = [r for r in eval_rows if r["group"] == group]
    sample.extend(rng.sample(pool, 10))

rng.shuffle(sample)

label_sheet = output_dir / "manual_review.md"

if not label_sheet.exists():
    lines = [
        "# Warranty Review Labels\n",
        "1 = warranty-related | 0 = unrelated\n"
    ]

    for i, row in enumerate(sample, 1):
        text = row["text"].replace("\n", " ").strip()
        lines.append(
            f"\n### {i}. ID: {row['review_id']} | label: ?\n\n{text}\n"
        )

    label_sheet.write_text("\n".join(lines), encoding="utf-8")

print("Evaluation reviews:", len(eval_rows))
print("Reviews to label:", len(sample))
print("Label file:", label_sheet)

Evaluation reviews: 90
Reviews to label: 30
Label file: C:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\outputs_warranty\manual_review.md


In [43]:
from pathlib import Path
from collections import defaultdict
import csv
import re

output_dir = Path("outputs_warranty")
if not output_dir.exists():
    output_dir = next(
        p / "outputs_warranty"
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "outputs_warranty").exists()
    )

def load_csv(name):
    with (output_dir / name).open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

scores = load_csv("day5_sbert_scores.csv")
w2v = load_csv("day5_word2vec_candidates_REVIEW_REQUIRED.csv")
fasttext = load_csv("day5_fasttext_candidates.csv")

content = (output_dir / "manual_review.md").read_text(encoding="utf-8")
labels = {
    rid: int(value)
    for rid, value in re.findall(
        r"### \d+\. ID: (\d+) \| label: ([01])", content
    )
}

def get_terms(rows, score_col):
    grouped = defaultdict(list)
    for row in rows:
        grouped[row["seed"]].append(row)

    terms = {"warranty", "guarantee"}
    for group in grouped.values():
        top = sorted(group, key=lambda r: float(r[score_col]), reverse=True)[:15]
        terms.update(row["candidate"].lower() for row in top)
    return terms

def make_regex(terms):
    return re.compile(
        r"\b(?:" + "|".join(re.escape(w) for w in sorted(terms)) + r")\b",
        re.I
    )

patterns = {
    "Regex": make_regex({"warranty", "guarantee"}),
    "Word2Vec": make_regex(get_terms(w2v, "cosine_similarity")),
    "FastText": make_regex(get_terms(fasttext, "similarity"))
}

def evaluate(y_true, y_pred):
    tp = sum(t == 1 and p == 1 for t, p in zip(y_true, y_pred))
    fp = sum(t == 0 and p == 1 for t, p in zip(y_true, y_pred))
    fn = sum(t == 1 and p == 0 for t, p in zip(y_true, y_pred))

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    return precision, recall, f1

labeled = [r for r in scores if r["review_id"] in labels]
y_true = [labels[r["review_id"]] for r in labeled]

print(f"Labeled: {len(labeled)}/30")
print(f"Positive: {sum(y_true)} | Negative: {len(y_true) - sum(y_true)}")

if len(labeled) < 30 or len(set(y_true)) < 2:
    print("Complete all 30 labels, including positive and negative reviews.")
else:
    comparison = []

    for name in ["Regex", "Word2Vec", "FastText", "Sentence-BERT"]:
        if name == "Sentence-BERT":
            predictions = [
                int(float(r["sbert_score"]) >= 0.30) for r in labeled
            ]
        else:
            predictions = [
                int(bool(patterns[name].search(r["text"]))) for r in labeled
            ]

        p, recall, f1 = evaluate(y_true, predictions)
        comparison.append({
            "model": name,
            "precision": round(p, 3),
            "recall": round(recall, 3),
            "f1": round(f1, 3)
        })

    comparison.sort(key=lambda r: r["f1"], reverse=True)

    for row in comparison:
        print(row)

    with (output_dir / "model_comparison.csv").open(
        "w", newline="", encoding="utf-8"
    ) as f:
        writer = csv.DictWriter(f, fieldnames=comparison[0].keys())
        writer.writeheader()
        writer.writerows(comparison)

    print("Saved: model_comparison.csv")

Labeled: 30/30
Positive: 9 | Negative: 21
{'model': 'Regex', 'precision': 0.8, 'recall': 0.889, 'f1': 0.842}
{'model': 'FastText', 'precision': 0.643, 'recall': 1.0, 'f1': 0.783}
{'model': 'Word2Vec', 'precision': 0.429, 'recall': 1.0, 'f1': 0.6}
{'model': 'Sentence-BERT', 'precision': 0, 'recall': 0.0, 'f1': 0}
Saved: model_comparison.csv


In [35]:
import re

content = (output_dir / "manual_review.md").read_text(encoding="utf-8")

print("Labeled:", len(re.findall(r"\| label: [01]\b", content)))
print("Remaining:", len(re.findall(r"\| label: \?", content)))

Labeled: 0
Remaining: 30


In [36]:
from pathlib import Path

root = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "outputs_warranty").is_dir()
)

report = """# Warranty Review Analysis

## Dataset
Train_subset_50k.csv — 50,000 preprocessed reviews.

## Keyword Baseline
- Keywords: warranty, guarantee
- Clean text matches: 965
- Full text matches: 979
- Additional summary matches: 14
- Products with matched reviews: 903

## Word Embeddings
- Word2Vec: 70 suggestions, 68 unique terms
- FastText: 60 suggestions, 52 unique terms
- Sentence-BERT: 90 reviews scored
- Co-occurrence analysis: 61 terms
- Spelling variants: 26 additional matches

## Findings
- FastText retrieved spelling variants such as warrenty and warrantee.
- Some embedding suggestions are unrelated to warranty.
- Cosine similarity and co-occurrence do not prove relevance.
- Full text is preferred for the keyword baseline.

## Status
- Regex baseline: completed
- Word2Vec candidate extraction: completed
- FastText experiment: completed
- Sentence-BERT scoring: completed
- Manual labeling: pending
- Precision, recall and F1 comparison: pending
- Final model selection: pending

## Next
Manually label the evaluation sample and compare all methods
using the same ground-truth labels.
"""

notes = root / "docs" / "WARRANTY_BASELINE_NOTES.md"
notes.parent.mkdir(exist_ok=True)
notes.write_text(report, encoding="utf-8")

print("Report updated:", notes)
print("Ready for Git.")

Report updated: C:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\docs\WARRANTY_BASELINE_NOTES.md
Ready for Git.


## Expanded Warranty Matching

Extend the baseline with spelling variants and compare the results.

In [37]:
import re
from collections import Counter

baseline_terms = [
    "warranty",
    "guarantee"
]

expanded_terms = baseline_terms + [
    "warranties",
    "warrantee",
    "warrantees",
    "warrenty"
]

def build_pattern(terms):
    return re.compile(
        r"\b(?:" + "|".join(map(re.escape, terms)) + r")\b",
        re.I
    )

baseline_regex = build_pattern(baseline_terms)
expanded_regex = build_pattern(expanded_terms)

baseline_ids = set()
expanded_ids = set()
new_terms = Counter()

for i, review in enumerate(reviews):
    text = review.get("full_text") or ""

    if baseline_regex.search(text):
        baseline_ids.add(i)

    match = expanded_regex.search(text)

    if match:
        expanded_ids.add(i)

        if i not in baseline_ids:
            new_terms[match.group().lower()] += 1

print("Baseline:", len(baseline_ids))
print("Expanded:", len(expanded_ids))
print("New matches:", len(expanded_ids - baseline_ids))
print("New terms:", dict(new_terms))

Baseline: 979
Expanded: 1010
New matches: 31
New terms: {'warranties': 10, 'warrenty': 8, 'warrantee': 13}


In [38]:
from collections import defaultdict
import csv

product_stats = defaultdict(list)

for i in expanded_ids:
    review = reviews[i]
    asin = review.get("asin", "").strip()

    try:
        rating = float(review["overall"])
    except (ValueError, TypeError):
        continue

    if asin:
        product_stats[asin].append(rating)

final_products = []

for asin, ratings in product_stats.items():
    final_products.append({
        "asin": asin,
        "warranty_reviews": len(ratings),
        "avg_rating": round(sum(ratings) / len(ratings), 2)
    })

final_products.sort(
    key=lambda row: row["warranty_reviews"],
    reverse=True
)

final_path = output_dir / "expanded_warranty_product_stats.csv"

with final_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["asin", "warranty_reviews", "avg_rating"]
    )
    writer.writeheader()
    writer.writerows(final_products)

print("Products:", len(final_products))
print("Saved:", final_path)

for row in final_products[:10]:
    print(row)

Products: 929
Saved: c:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\outputs_warranty\expanded_warranty_product_stats.csv
{'asin': 'B00UB9UJFQ', 'warranty_reviews': 3, 'avg_rating': 4.67}
{'asin': 'B01A6PGZ70', 'warranty_reviews': 3, 'avg_rating': 5.0}
{'asin': 'B007PJ4Q4A', 'warranty_reviews': 3, 'avg_rating': 1.0}
{'asin': 'B007B4UKOE', 'warranty_reviews': 3, 'avg_rating': 3.67}
{'asin': 'B013HNYVCE', 'warranty_reviews': 3, 'avg_rating': 3.33}
{'asin': 'B00TKFEE9E', 'warranty_reviews': 3, 'avg_rating': 2.67}
{'asin': 'B016QCPNUW', 'warranty_reviews': 3, 'avg_rating': 2.33}
{'asin': 'B01DOJDSGQ', 'warranty_reviews': 3, 'avg_rating': 4.33}
{'asin': 'B00LF10KTE', 'warranty_reviews': 3, 'avg_rating': 5.0}
{'asin': 'B00IVPU7AO', 'warranty_reviews': 3, 'avg_rating': 1.67}


In [39]:
# Inspect newly detected warranty reviews

new_review_ids = sorted(expanded_ids - baseline_ids)

for n, i in enumerate(new_review_ids, 1):
    review = reviews[i]
    text = review.get("full_text") or ""

    match = expanded_regex.search(text)

    start = max(0, match.start() - 90)
    end = min(len(text), match.end() + 140)

    context = text[start:end].replace("\n", " ")

    print(f"\n{n}. ASIN: {review['asin']}")
    print(f"Term: {match.group()}")
    print(f"Rating: {review['overall']}")
    print(f"Context: ...{context}...")
    print("-" * 60)


1. ASIN: B00YBCG3YK
Term: warranties
Rating: 5
Context: ...dreds of dollars on will last long enough to make the purchase worth it. The Square Trade warranties offered through Amazon are great because I no longer have to worry about replacing something fairly new due to it failing. I haven't had to...
------------------------------------------------------------

2. ASIN: B00BR5FZ7U
Term: warranties
Rating: 5
Context: ...the last few years about there being more resistance on Squaretrade's part to honor their warranties and I hope that is not the case. I continue to purchase them, occasionally, as needed, based on my positive prior experiences. I do have to...
------------------------------------------------------------

3. ASIN: B00MEYSZWU
Term: warrenty
Rating: 1
Context: ...my bad should have called the product was purchased by amazon 2013 there is no warrenty. The charge port has not worked since day 1. my bad should have called.. I did call tonight and the fee to repair is 267.00 

In [41]:
import csv
import json
from collections import Counter

# Save expanded warranty matches
final_matches = []

for i in sorted(expanded_ids):
    review = reviews[i]
    text = review.get("full_text") or ""
    match = expanded_regex.search(text)

    final_matches.append({
        "asin": review["asin"],
        "overall": review["overall"],
        "matched_term": match.group().lower(),
        "full_text": text
    })

matches_path = output_dir / "expanded_warranty_matches.csv"

with matches_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["asin", "overall", "matched_term", "full_text"]
    )
    writer.writeheader()
    writer.writerows(final_matches)

# Save summary
summary = {
    "total_reviews": len(reviews),
    "baseline_matches": len(baseline_ids),
    "expanded_matches": len(expanded_ids),
    "additional_matches": len(expanded_ids - baseline_ids),
    "products": len(final_products),
    "keywords": expanded_terms,
    "validation": "Preliminary - manual evaluation pending"
}

summary_path = output_dir / "expanded_warranty_summary.json"

summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("=== Warranty Analysis ===")
print("Total reviews:", len(reviews))
print("Baseline:", len(baseline_ids))
print("Expanded:", len(expanded_ids))
print("Additional:", len(expanded_ids - baseline_ids))
print("Products:", len(final_products))

print("\nFiles saved:")
print(matches_path.name)
print("expanded_warranty_product_stats.csv")
print(summary_path.name)

=== Warranty Analysis ===
Total reviews: 50000
Baseline: 979
Expanded: 1010
Additional: 31
Products: 929

Files saved:
expanded_warranty_matches.csv
expanded_warranty_product_stats.csv
expanded_warranty_summary.json


In [42]:
import re

content = (output_dir / "manual_review.md").read_text(encoding="utf-8")

labels = re.findall(
    r"### \d+\. ID: (\d+) \| label: ([01])",
    content
)

print("Labeled reviews:", len(labels))
print("Positive:", sum(int(label) for _, label in labels))
print("Negative:", sum(label == "0" for _, label in labels))

Labeled reviews: 30
Positive: 9
Negative: 21


In [44]:
import re
import csv
from statistics import mean

# Load evaluation data
with (output_dir / "day5_sbert_scores.csv").open(
    encoding="utf-8-sig", newline=""
) as f:
    eval_data = list(csv.DictReader(f))

# Load manual labels
content = (output_dir / "manual_review.md").read_text(
    encoding="utf-8"
)

labels = {
    review_id: int(label)
    for review_id, label in re.findall(
        r"### \d+\. ID: (\d+) \| label: ([01])",
        content
    )
}

labeled = [
    row for row in eval_data
    if row["review_id"] in labels
]

# Expanded Regex
terms = [
    "warranty",
    "guarantee",
    "warranties",
    "warrantee",
    "warrantees",
    "warrenty"
]

expanded_pattern = re.compile(
    r"\b(?:" + "|".join(map(re.escape, terms)) + r")\b",
    re.I
)

y_true = [labels[r["review_id"]] for r in labeled]

y_pred = [
    int(bool(expanded_pattern.search(r["text"])))
    for r in labeled
]

# Calculate metrics
tp = sum(t == 1 and p == 1 for t, p in zip(y_true, y_pred))
fp = sum(t == 0 and p == 1 for t, p in zip(y_true, y_pred))
fn = sum(t == 1 and p == 0 for t, p in zip(y_true, y_pred))

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

print("=== Expanded Regex ===")
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1:", round(f1, 3))
print(f"TP: {tp} | FP: {fp} | FN: {fn}")

# Inspect Sentence-BERT scores
positive_scores = [
    float(r["sbert_score"])
    for r in labeled
    if labels[r["review_id"]] == 1
]

negative_scores = [
    float(r["sbert_score"])
    for r in labeled
    if labels[r["review_id"]] == 0
]

print("\n=== Sentence-BERT Scores ===")
print("Positive mean:", round(mean(positive_scores), 3))
print("Negative mean:", round(mean(negative_scores), 3))
print("Positive range:", round(min(positive_scores), 3),
      "-", round(max(positive_scores), 3))
print("Negative range:", round(min(negative_scores), 3),
      "-", round(max(negative_scores), 3))

=== Expanded Regex ===
Precision: 0.8
Recall: 0.889
F1: 0.842
TP: 8 | FP: 2 | FN: 1

=== Sentence-BERT Scores ===
Positive mean: 0.19
Negative mean: 0.114
Positive range: 0.088 - 0.291
Negative range: -0.053 - 0.268


In [45]:
import csv
import json

# Load model comparison
comparison_path = output_dir / "model_comparison.csv"

with comparison_path.open(
    encoding="utf-8-sig", newline=""
) as f:
    comparison = list(csv.DictReader(f))

# Add expanded regex
comparison = [
    row for row in comparison
    if row["model"] != "Expanded Regex"
]

comparison.append({
    "model": "Expanded Regex",
    "precision": 0.800,
    "recall": 0.889,
    "f1": 0.842
})

comparison.sort(
    key=lambda row: float(row["f1"]),
    reverse=True
)

# Save comparison
with comparison_path.open(
    "w", newline="", encoding="utf-8"
) as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["model", "precision", "recall", "f1"]
    )
    writer.writeheader()
    writer.writerows(comparison)

# Save final summary
final_summary = {
    "dataset_size": 50000,
    "baseline_reviews": 979,
    "expanded_reviews": 1010,
    "additional_reviews": 31,
    "products": 929,
    "evaluation_sample": 30,
    "positive_labels": 9,
    "negative_labels": 21,
    "selected_method": "Expanded Regex",
    "evaluation_f1": 0.842,
    "limitations": [
        "Preliminary manual labels",
        "Small, stratified evaluation sample",
        "Sentence-BERT threshold not calibrated",
        "Full dataset not evaluated"
    ]
}

summary_path = output_dir / "warranty_final_evaluation.json"

summary_path.write_text(
    json.dumps(final_summary, indent=2),
    encoding="utf-8"
)

print("Final comparison:")
for row in comparison:
    print(row)

print("\nSaved:")
print(comparison_path.name)
print(summary_path.name)

Final comparison:
{'model': 'Regex', 'precision': '0.8', 'recall': '0.889', 'f1': '0.842'}
{'model': 'Expanded Regex', 'precision': 0.8, 'recall': 0.889, 'f1': 0.842}
{'model': 'FastText', 'precision': '0.643', 'recall': '1.0', 'f1': '0.783'}
{'model': 'Word2Vec', 'precision': '0.429', 'recall': '1.0', 'f1': '0.6'}
{'model': 'Sentence-BERT', 'precision': '0', 'recall': '0.0', 'f1': '0'}

Saved:
model_comparison.csv
warranty_final_evaluation.json


In [46]:
import csv
import json
import re
from pathlib import Path

# Load saved outputs
def read_csv(filename):
    with (output_dir / filename).open(
        encoding="utf-8-sig", newline=""
    ) as f:
        return list(csv.DictReader(f))

matches = read_csv("expanded_warranty_matches.csv")
products = read_csv("expanded_warranty_product_stats.csv")
comparison = read_csv("model_comparison.csv")

summary = json.loads(
    (output_dir / "expanded_warranty_summary.json").read_text(
        encoding="utf-8"
    )
)

# Check saved results
assert len(matches) == summary["expanded_matches"]
assert len(products) == summary["products"]
assert len(matches) == 1010
assert len(products) == 929

# Check manual labels
label_text = (output_dir / "manual_review.md").read_text(
    encoding="utf-8"
)

labels = re.findall(
    r"### \d+\. ID: (\d+) \| label: ([01])",
    label_text
)

assert len(labels) == 30
assert len({review_id for review_id, _ in labels}) == 30

# Check model comparison
model_names = {row["model"] for row in comparison}

expected_models = {
    "Regex",
    "Expanded Regex",
    "Word2Vec",
    "FastText",
    "Sentence-BERT"
}

assert model_names == expected_models

print("=== FINAL VALIDATION ===")
print("Reviews:", len(matches))
print("Products:", len(products))
print("Labeled samples:", len(labels))
print("Compared methods:", len(model_names))
print("Status: ALL CHECKS PASSED")

=== FINAL VALIDATION ===
Reviews: 1010
Products: 929
Labeled samples: 30
Compared methods: 5
Status: ALL CHECKS PASSED


In [47]:
# Save final warranty report

from pathlib import Path
import csv
import json

def load_csv(name):
    with (output_dir / name).open(
        encoding="utf-8-sig", newline=""
    ) as f:
        return list(csv.DictReader(f))

results = load_csv("model_comparison.csv")
summary = json.loads(
    (output_dir / "expanded_warranty_summary.json").read_text(
        encoding="utf-8"
    )
)

docs_dir = output_dir.parent / "docs"
docs_dir.mkdir(exist_ok=True)

table_rows = "\n".join(
    f"| {r['model']} | {float(r['precision']):.3f} | "
    f"{float(r['recall']):.3f} | {float(r['f1']):.3f} |"
    for r in results
)

report = f"""# Warranty Review Analysis — Final Report

## Overview
Analyzed 50,000 preprocessed Amazon electronics reviews
to identify warranty-related feedback.

## Approach
- Built a Regex baseline using `warranty` and `guarantee`.
- Explored related terms using Word2Vec and FastText.
- Checked spelling variants and candidate contexts.
- Tested Sentence-BERT semantic similarity.
- Extended Regex with selected spelling variants.

## Results
- Baseline matches: {summary['baseline_matches']}
- Expanded matches: {summary['expanded_matches']}
- Additional matches: {summary['additional_matches']}
- Products identified: {summary['products']}

## Preliminary Evaluation

30 manually reviewed samples: 9 positive and 21 negative.

| Method | Precision | Recall | F1 |
|---|---:|---:|---:|
{table_rows}

## Selected Approach
Expanded Regex was selected as a practical preliminary approach.
It covers additional spelling variants without reducing F1
on the evaluation sample.

## Limitations
- Evaluation uses only 30 purposefully sampled reviews.
- Initial labels require independent verification.
- The 31 additional matches were not all manually verified.
- Sentence-BERT used an uncalibrated similarity threshold.
- Results are based on the 50k subset, not the full dataset.
- Overall ratings approximate warranty satisfaction;
  they do not directly measure warranty-specific sentiment.

## Outputs
- `expanded_warranty_matches.csv`
- `expanded_warranty_product_stats.csv`
- `expanded_warranty_summary.json`
- `model_comparison.csv`
- `warranty_final_evaluation.json`
"""

report_path = docs_dir / "WARRANTY_FINAL_REPORT.md"
report_path.write_text(report, encoding="utf-8")

print("Report saved:", report_path)
print("Warranty analysis ready for team review.")

Report saved: c:\Users\HP\Desktop\Projects\Learn\ML&AI\project2\incident-sentiment-analyzer\docs\WARRANTY_FINAL_REPORT.md
Warranty analysis ready for team review.
